# Perpetual futures market data smoke test

This notebook exercises the consumer-facing `MarketDataService` flow: configure an instrument and date range, load it, then inspect the registry and downloaded CSV. Provider selection stays inside the loader layer.

The live example uses Tardis's documented Binance USDT Futures BTCUSDT trades sample for 2020-02-01. Tardis makes first-of-month datasets available without an API key; set `TARDIS_API_KEY` in the notebook environment if you want authenticated access for other dates.

In [ ]:
from datetime import date
from pathlib import Path
import csv
import gzip
import sys
import tempfile

# Support running from either the workspace root or this package's notebook folder.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'src' / 'quant_common').is_dir():
        sys.path.insert(0, str(candidate / 'src'))
        break
    if (candidate / 'quant-common' / 'src' / 'quant_common').is_dir():
        sys.path.insert(0, str(candidate / 'quant-common' / 'src'))
        break

from quant_common.data import DataConfig, MarketDataService
from quant_common.data.cache import DatasetCache
from quant_common.data.instruments import PerpetualFuture

## Verify the loader and registry offline

This section seeds a small cached CSV and asks the service to load it. It exercises the service, cache lookup, and registry without calling a provider or using API quota.

In [ ]:
btc_perpetual = PerpetualFuture(
    exchange='binance-futures',
    symbol='BTCUSDT',
    data_types=('trades',),
)
offline_root = Path(tempfile.mkdtemp(prefix='quant-common-market-data-'))
offline_cache = DatasetCache(offline_root)
offline_day = date(2020, 2, 1)
offline_path = offline_cache.path_for(btc_perpetual.key, 'trades', offline_day)
offline_path.parent.mkdir(parents=True, exist_ok=True)
with gzip.open(offline_path, mode='wt', newline='') as cached_csv:
    cached_csv.write('timestamp,symbol,side,price,amount\n')
    cached_csv.write('1580515200000000,BTCUSDT,buy,9300.0,0.01\n')

offline_service = MarketDataService(
    instruments=[btc_perpetual],
    start_date=offline_day,
    end_date=date(2020, 2, 2),
    config=DataConfig(storage_path=str(offline_root)),
)
offline_registry = offline_service.load()
offline_dataset = offline_registry.get(btc_perpetual)
assert offline_dataset.instrument_key == btc_perpetual.key
assert offline_dataset.files_by_data_type['trades'] == (offline_path,)
print('Offline cache hit and registry lookup succeeded:', offline_registry.keys())

## Load the live Tardis sample

The live example uses Tardis's documented Binance USDT Futures BTCUSDT trades sample for 2020-02-01. Tardis makes first-of-month datasets available without an API key; set `TARDIS_API_KEY` in the notebook environment if using a different date.

In [ ]:
btc_perpetual = PerpetualFuture(
    exchange='binance-futures',
    symbol='BTCUSDT',
    data_types=('trades',),
)

market_data = MarketDataService(
    instruments=[btc_perpetual],
    start_date=date(2020, 2, 1),
    end_date=date(2020, 2, 2),  # exclusive
)
registry = market_data.load()
dataset = registry.get(btc_perpetual)

print('Registry keys:', registry.keys())
print('Instrument key:', btc_perpetual.key)
print('Coverage:', dataset.start_date, 'to', dataset.end_date, '(end exclusive)')
print('Files by data type:', dataset.files_by_data_type)

## Inspect the loaded CSV

The loader stores daily Tardis CSV datasets in gzip format. This cell reads a few rows without loading the full dataset into memory.

In [ ]:
trade_file = dataset.files_by_data_type['trades'][0]
print(f'Cached file: {trade_file} ({trade_file.stat().st_size:,} bytes)')
with gzip.open(trade_file, mode='rt', newline='') as compressed_csv:
    trade_rows = csv.DictReader(compressed_csv)
    first_row = next(trade_rows, None)

print('CSV columns:', trade_rows.fieldnames)
print('First row:', first_row if first_row else '(no trades for this day)')

assert dataset.instrument_key == btc_perpetual.key
assert trade_file.exists()
assert trade_file.stat().st_size > 0
assert first_row is not None

## Confirm cache reuse

Loading the same instrument and date range again should reuse the cached file. No provider configuration is needed in the consumer call.

In [ ]:
cached_path_before = trade_file.resolve()
second_registry = market_data.load()
cached_path_after = second_registry.get(btc_perpetual).files_by_data_type['trades'][0].resolve()
assert cached_path_before == cached_path_after
print('Cache reuse confirmed:', cached_path_after)